# 05. Entradas do Modelo 2 (RGP‑t)

Etapa 5 do fluxo: `01_download.ipynb` → `02_serie_modelos.ipynb` → `03_guia_bases_modelos.ipynb` → `04_entradas_modelo1_gp.ipynb` / `05_entradas_modelo2_rgp.ipynb` / `06_entradas_modelo3_gplvm.ipynb` → notebooks dos modelos (`07`–`09`).

Gera e salva a entrada do RGP‑t a partir da `serie` salva no `02_serie_modelos.ipynb`. **Não ajusta o modelo.** O RGP‑t recebe todas as estações juntas, como sequências completas e ordenadas; dias sem valor visível viram 0 e a máscara manda o modelo ignorá-los. Detalhes na seção 4.2 do `03_guia_bases_modelos.ipynb`.

Saída: `data/exports/<NOME>/modelo2_rgp/entradas.parquet` (uma linha por estação e dia) e `config.json`.

## 1. Carregar os módulos

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root(start: Path) -> Path:
    for folder in (start.resolve(), *start.resolve().parents):
        if (folder / "src" / "main.py").is_file() and (folder / "data" / "usgs").is_dir():
            return folder
    raise FileNotFoundError("Não encontrei src/main.py e data/usgs. Abra o notebook na pasta do projeto.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import EXPORT_ROOT
from src.model_data import split_summary
from src.model_exports import rgp_table, rgp_arrays, save_inputs

## 2. Ler a base

In [ ]:
NOME = "serie_2010_2025_200est"   # o mesmo NOME do 02_serie_modelos.ipynb

serie = pd.read_parquet(EXPORT_ROOT / f"{NOME}.parquet")
print("Base:", (EXPORT_ROOT / f"{NOME}_config.json").read_text(encoding="utf-8"))
print(f"{len(serie):,} linhas | {serie['sensor_id'].nunique()} estações")

## 3. Montar e conferir a entrada

Uma linha por (estação, dia), com as 200 estações:

| Coluna | Conteúdo | Uso no RGP‑t |
|---|---|---|
| `y_entrada` | `y` nos dias visíveis, 0 nos outros | valor lido pela RNN |
| `visivel` | 1 nos dias visíveis | máscara: só esses dias entram no ELBO (no treino) e no escore |
| `escondido_aval` | 1 nos dias escondidos | onde se mede a reconstrução |
| `y_true` | valor verdadeiro (NaN sem valor) | **só para a métrica** |

A sequência é sempre lida inteira; os blocos (`split`) só decidem quais dias entram no ajuste ou em cada métrica.

In [ ]:
entradas = rgp_table(serie)

rgp = rgp_arrays(entradas)
print("estações (linhas):", len(rgp["sensors"]), "| dias (colunas):", rgp["y"].shape[1])
print("rnn_in", rgp["rnn_in"].shape, " NaN:", np.isnan(rgp["rnn_in"]).any())
display(pd.DataFrame({
    "dias no ajuste/escore": {b: int(rgp["split_masks"][b].sum()) for b in rgp["split_masks"]},
    "dias escondidos": {b: int(rgp["eval_masks"][b].sum()) for b in rgp["eval_masks"]},
}))

## 4. Salvar

Se a entrada já existir com as mesmas escolhas, nada é regravado. Se as escolhas mudarem, a célula para com erro: apague a pasta `data/exports/<NOME>/modelo2_rgp/` para gerar de novo.

In [ ]:
save_inputs(entradas, NOME, "modelo2_rgp", config={
    "estacoes": rgp["sensors"],   # na ordem das linhas das matrizes (S, T)
})

## 5. Como o notebook do modelo lê esta entrada

```python
from src.model_exports import load_inputs, rgp_arrays

entradas, cfg = load_inputs("serie_2010_2025_200est", "modelo2_rgp")
rgp = rgp_arrays(entradas)

X = rgp["rnn_in"]                                  # (S, T, 2) = [valor, máscara]
M_train = rgp["fit_mask"]                          # dias que entram no ELBO
M_test, H_test = rgp["split_masks"]["teste"], rgp["eval_masks"]["teste"]
Y_true = rgp["y_true"]                             # só para a métrica
```

Para usar menos estações, filtre antes de converter: `rgp_arrays(entradas[entradas["sensor_id"].isin(lista)])`.